# 추천 시스템: 순위
---

1. 검색 단계: 가능한 모든 후보 중 수백개의 초기 후보세트를 선택합니다. 주요목적은 사용자가 관심없는 모든 후보를 효율적으로 제거하는 것입니다. 계산효율성이 좋아야한다.
2. 순위 단계:검색 모델의 출력을 가져와 미세조정하여 가능한 한 최고의 권장사항을 선택합니다. 그 작업은 사용자가 관심을 가질 만한 항목 집합을 가능한 후보의 최종 목록으로 좁히는 것입니다.

In [5]:
import os
import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_recommenders as tfrs

In [25]:
ratings = tfds.load("movielens/100k-ratings", split="train")
# 이번에는 ratings만 가져오네.


ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
    "user_rating": x["user_rating"]
})

for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'user_id': b'138',
 'user_rating': 4.0}


2026-03-12 14:48:43.388531: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [18]:
print(len(ratings))

100000


In [19]:
tf.random.set_seed(42)
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

trian = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [37]:
# 데이터에 있는 고유한 user_id와 movie_title
movie_titles = ratings.batch(1_000_000).map(lambda x: x["movie_title"])
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:5]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)'],
      dtype=object)

---

리스트와 배치의 차이

.batch(1000) - 하는 이유는 기본적으로 배치 안 쓰면 하나씩 뽑아쓰게됨. 크게 배치수를하면 한번에 가져올 수 있겠지. 그리고 데이터셋에서 텐서상자가 씌어진 형태가 된다.
> Tensor(1000개) → Tensor(1000개) → Tensor(1000개)

list(movie_titles) - 상자안의 상자
> [ Tensor(1000개) → Tensor(1000개) → Tensor(1000개) ]

np.concatenate(...) - 하나의 긴 줄로 이어붙임. (리스트도 배치 벽도 허묾. 결국 하나의 거대한 넘파이 배열이 탄생)

> [ 3000개 짜리 ] 

---
**데이터셋은 {키: 값}딕셔너리를 한 번에 한 줄씩 줄줄이 흐르게 하는 장치**

    사실 딕셔너리가아닌 텐서들이 들어가있다. 딕셔너리처럼보이는텐서들의 행렬임.

- 데이터셋-> 리스트 묶기 (너무 느리미. 왜냐면 데이터셋은 한번에 한줄이니까..데이터가 너무많으면..으악!)
- 데이터셋-> 배치로 묶기 -> 리스트로 묶기 (정석)

---
10000개 있는 데이터를 

만약 배치를1000으로 하면 shape=(1000,) 이렇게 나온다. shape은 철저히 하나상자의 모형을 알려줌.

배치를 안 했을 때는 상자가 없으니 걍 ()